# Backfill `duration` for Videos Missing It

Many media records have `duration = NULL` because the upload form didn't always extract duration client-side. This notebook reads the MP4/MOV `mvhd` atom from R2 to fill in the missing values.

### How it works

The `mvhd` (movie header) atom inside every MP4/MOV file contains:
- **timescale**: units per second (e.g. 600)
- **duration**: total length in timescale units
- `duration_seconds = duration / timescale`

We use efficient **range requests** (~5KB per file) to read just the atom headers — no need to download full videos.

### Steps

1. **Connect** to Supabase + R2
2. **Find** all media with `duration IS NULL`
3. **Extract** duration from `mvhd` atom via R2 range requests
4. **Review** results + export to CSV
5. **Apply** updates to Supabase

### Safe to re-run

On re-runs, successful extractions are loaded from the previous `duration_report.csv` — only failures are retried.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

def clean_env(key):
    """Get env var and strip any inline # comments that dotenv may include."""
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

missing = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing:
    print(f"Missing config: {', '.join(missing)}")
else:
    print("All config loaded.")

All config loaded.


In [2]:
import boto3
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

r2 = boto3.client(
    service_name="s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


---
## Step 1: Find Videos Missing Duration

In [3]:
from tabulate import tabulate

# Fetch all media records
all_media = []
offset = 0
while True:
    res = sb.table("media").select(
        "id, title, storage_path, original_filename, duration, media_type, file_size_bytes"
    ).range(offset, offset + 1000 - 1).execute()
    all_media.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

# Split into videos with and without duration
has_duration = [r for r in all_media if r.get("duration") is not None]
missing_duration = [
    r for r in all_media
    if r.get("duration") is None and r.get("storage_path")
]
no_storage_path = [
    r for r in all_media
    if r.get("duration") is None and not r.get("storage_path")
]

print(f"Total media records: {len(all_media)}")
print(f"  Already have duration:  {len(has_duration)}")
print(f"  Missing duration (video in R2): {len(missing_duration)}")
print(f"  Missing duration (no storage_path — images?): {len(no_storage_path)}")

if missing_duration:
    print(f"\n{'─' * 80}")
    print(f"VIDEOS MISSING DURATION (first 30)")
    print(f"{'─' * 80}")
    rows = []
    for r in sorted(missing_duration, key=lambda x: x.get("original_filename", ""))[:30]:
        size_mb = f"{r['file_size_bytes'] / 1024 / 1024:.1f} MB" if r.get("file_size_bytes") else "—"
        rows.append([
            r.get("original_filename", "—")[:40],
            r.get("media_type", "—"),
            size_mb,
            r["storage_path"][:45],
        ])
    print(tabulate(rows, headers=["Filename", "Type", "Size", "Storage Path"], tablefmt="simple"))
    if len(missing_duration) > 30:
        print(f"\n  ... and {len(missing_duration) - 30} more.")

Total media records: 359
  Already have duration:  214
  Missing duration (video in R2): 145
  Missing duration (no storage_path — images?): 0

────────────────────────────────────────────────────────────────────────────────
VIDEOS MISSING DURATION (first 30)
────────────────────────────────────────────────────────────────────────────────
Filename                          Type    Size      Storage Path
--------------------------------  ------  --------  ---------------------------------------------
2024_06_02_20_44_34_IMG_0462.MOV  video   15.1 MB   videos/cc2dbed2-2b51-45a8-bb98-9e3f3c990185.M
2024_06_02_22_49_11_IMG_0463.MOV  video   275.4 MB  videos/88a40332-91b6-4a0d-a93f-a0dd375f9a33.M
2024_07_01_21_53_50_IMG_0847.MP4  video   38.1 MB   videos/02186384-d567-4bc6-b43a-4c12d8c0806f.M
2024_07_04_10_14_14_IMG_0911.MP4  video   185.7 MB  videos/a973886b-7aeb-44f3-96ce-44b2c83b38e1.M
2024_07_09_19_54_25_IMG_0975.MOV  video   28.2 MB   videos/52903668-a297-40b4-b9dc-6aba47ef8e25.M
2024_0

---
## Step 2: MP4/MOV Duration Parser

Reads the `mvhd` atom to extract `timescale` and `duration` fields.

**mvhd layout (version 0):**
```
[4 bytes size][4 bytes 'mvhd'][1 byte version][3 bytes flags]
[4 bytes creation_time][4 bytes modification_time]
[4 bytes timescale][4 bytes duration]
```

**mvhd layout (version 1):** same but creation/modification/duration are 8 bytes each.

In [4]:
import struct


def parse_mvhd_duration(data: bytes) -> float | None:
    """
    Search raw bytes for the 'mvhd' atom and extract duration in seconds.
    Returns duration as float (seconds) or None.
    """
    idx = data.find(b'mvhd')
    if idx == -1:
        return None

    # mvhd layout: [4 bytes 'mvhd'][1 byte version][3 bytes flags]
    #   version 0: [4B creation_time][4B modification_time][4B timescale][4B duration]
    #   version 1: [8B creation_time][8B modification_time][4B timescale][8B duration]
    offset = idx + 4  # skip 'mvhd'
    if offset + 4 > len(data):
        return None

    version = data[offset]
    offset += 4  # skip version (1) + flags (3)

    if version == 0:
        # creation_time(4) + modification_time(4) + timescale(4) + duration(4) = 16 bytes
        if offset + 16 > len(data):
            return None
        offset += 8  # skip creation_time + modification_time
        timescale = struct.unpack('>I', data[offset:offset + 4])[0]
        offset += 4
        duration = struct.unpack('>I', data[offset:offset + 4])[0]
    elif version == 1:
        # creation_time(8) + modification_time(8) + timescale(4) + duration(8) = 28 bytes
        if offset + 28 > len(data):
            return None
        offset += 16  # skip creation_time + modification_time
        timescale = struct.unpack('>I', data[offset:offset + 4])[0]
        offset += 4
        duration = struct.unpack('>Q', data[offset:offset + 8])[0]
    else:
        return None

    if timescale == 0 or duration == 0:
        return None

    duration_seconds = duration / timescale

    # Sanity check: reject durations > 24 hours or <= 0
    if duration_seconds <= 0 or duration_seconds > 86400:
        return None

    return duration_seconds


def find_moov_offset(data: bytes, file_size: int) -> tuple[int, int] | None:
    """
    Walk top-level MP4/MOV atoms to find moov's (offset, size).
    data should be the first few KB of the file.
    """
    pos = 0
    while pos + 8 <= len(data):
        atom_size = struct.unpack('>I', data[pos:pos + 4])[0]
        atom_type = data[pos + 4:pos + 8]

        # Handle extended size (atom_size == 1 means 64-bit size follows)
        if atom_size == 1 and pos + 16 <= len(data):
            atom_size = struct.unpack('>Q', data[pos + 8:pos + 16])[0]

        # atom_size == 0 means "rest of file"
        if atom_size == 0:
            atom_size = file_size - pos

        if atom_size < 8:
            break  # corrupt

        if atom_type == b'moov':
            return (pos, atom_size)

        pos += atom_size

    return None


def get_duration_from_r2(storage_path: str, file_size: int | None = None) -> float | None:
    """
    Locate moov atom via atom table, then fetch just the mvhd portion.
    Falls back to scanning start/end chunks if atom walking fails.
    Returns duration in seconds (float) or None.
    """
    # Get file size if we don't have it
    if not file_size:
        try:
            head = r2.head_object(Bucket=R2_BUCKET_NAME, Key=storage_path)
            file_size = head.get("ContentLength")
        except Exception:
            pass

    # ── Strategy 1: Walk atom table to find moov precisely ──
    if file_size and file_size > 0:
        try:
            # Read first 4KB to walk top-level atoms
            resp = r2.get_object(
                Bucket=R2_BUCKET_NAME,
                Key=storage_path,
                Range="bytes=0-4095"
            )
            header_data = resp['Body'].read()

            moov = find_moov_offset(header_data, file_size)

            if moov:
                moov_offset, moov_size = moov
                # mvhd is the first sub-atom inside moov, we only need ~200 bytes
                fetch_size = min(1024, moov_size)
                resp = r2.get_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=storage_path,
                    Range=f"bytes={moov_offset}-{moov_offset + fetch_size - 1}"
                )
                moov_data = resp['Body'].read()
                result = parse_mvhd_duration(moov_data)
                if result:
                    return result

            # moov not in first 4KB of atom headers — check near end
            if file_size > 4096:
                tail_start = max(0, file_size - 4096)
                resp = r2.get_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=storage_path,
                    Range=f"bytes={tail_start}-{file_size - 1}"
                )
                tail_data = resp['Body'].read()

                moov_tail = find_moov_offset(tail_data, file_size - tail_start + tail_start)
                if moov_tail:
                    moov_abs_offset = tail_start + moov_tail[0]
                    fetch_size = min(1024, moov_tail[1])
                    resp = r2.get_object(
                        Bucket=R2_BUCKET_NAME,
                        Key=storage_path,
                        Range=f"bytes={moov_abs_offset}-{moov_abs_offset + fetch_size - 1}"
                    )
                    moov_data = resp['Body'].read()
                    result = parse_mvhd_duration(moov_data)
                    if result:
                        return result
        except Exception:
            pass

    # ── Strategy 2: Fallback — brute-force scan larger chunks ──
    CHUNK = 512 * 1024  # 512KB

    # Try start
    try:
        resp = r2.get_object(
            Bucket=R2_BUCKET_NAME,
            Key=storage_path,
            Range=f"bytes=0-{CHUNK - 1}"
        )
        data = resp['Body'].read()
        result = parse_mvhd_duration(data)
        if result:
            return result
    except Exception:
        pass

    # Try end
    if file_size and file_size > CHUNK:
        try:
            start = file_size - CHUNK
            resp = r2.get_object(
                Bucket=R2_BUCKET_NAME,
                Key=storage_path,
                Range=f"bytes={start}-{file_size - 1}"
            )
            data = resp['Body'].read()
            result = parse_mvhd_duration(data)
            if result:
                return result
        except Exception:
            pass

    return None


print("MP4/MOV duration parser ready.")

MP4/MOV duration parser ready.


---
## Step 3: Extract Durations from R2

Processes all videos missing duration. On re-runs, reuses successful extractions from `duration_report.csv`.

In [5]:
import csv as csv_mod
from pathlib import Path

REPORT_PATH = Path(r"C:\Users\minds\Videos\DANCE\dance-library\TESTING\duration_report.csv")

# ── Load previous report if it exists (reuse successful extractions) ──
prev_report = {}  # id -> {status, extracted_duration}
if REPORT_PATH.exists():
    with open(REPORT_PATH, "r", encoding="utf-8") as f:
        reader = csv_mod.DictReader(f)
        for row in reader:
            prev_report[row["id"]] = row
    prev_ready = sum(1 for r in prev_report.values() if r["status"] == "ready")
    prev_failed = sum(1 for r in prev_report.values() if r["status"] == "no_data")
    print(f"Loaded previous report: {len(prev_report)} records")
    print(f"  {prev_ready} ready (will reuse), {prev_failed} no_data (will retry)")
else:
    print("No previous report found — will process all records from scratch.")

# ── Categorize ──
from_prev = []   # reuse from previous successful extraction
needs_r2 = []    # needs fresh R2 extraction

for r in missing_duration:
    prev = prev_report.get(r["id"])
    if prev and prev["status"] == "ready" and prev.get("extracted_duration"):
        from_prev.append({**r, "extracted_duration": float(prev["extracted_duration"])})
    else:
        needs_r2.append(r)

print(f"\nReusing from previous report: {len(from_prev)}")
print(f"Need R2 extraction: {len(needs_r2)}")

No previous report found — will process all records from scratch.

Reusing from previous report: 0
Need R2 extraction: 145


In [6]:
# ── Extract duration from R2 for new/retry records ──
r2_results = []  # (record, duration_seconds_or_None)
extracted_count = 0
failed_count = 0

print(f"Extracting duration from {len(needs_r2)} files via R2...")
print()

for i, r in enumerate(needs_r2):
    storage_path = r["storage_path"]
    file_size = r.get("file_size_bytes")

    dur = get_duration_from_r2(storage_path, file_size)
    r2_results.append((r, dur))

    if dur:
        extracted_count += 1
    else:
        failed_count += 1

    if (i + 1) % 50 == 0 or i == len(needs_r2) - 1:
        print(f"  [{i + 1}/{len(needs_r2)}] extracted: {extracted_count}, failed: {failed_count}")

print(f"\nDone. Extracted: {extracted_count}, Failed: {failed_count}")

Extracting duration from 145 files via R2...

  [50/145] extracted: 50, failed: 0
  [100/145] extracted: 100, failed: 0
  [145/145] extracted: 145, failed: 0

Done. Extracted: 145, Failed: 0


---
## Step 4: Review Results

In [7]:
def fmt_duration(seconds: float) -> str:
    """Format seconds as H:MM:SS or M:SS."""
    s = int(seconds)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    if h > 0:
        return f"{h}:{m:02d}:{sec:02d}"
    return f"{m}:{sec:02d}"


successful_r2 = [(r, d) for r, d in r2_results if d]
failed_r2 = [(r, d) for r, d in r2_results if not d]

total_successful = len(from_prev) + len(successful_r2)

print("=" * 60)
print("DURATION EXTRACTION RESULTS")
print("=" * 60)
print(f"  Reused from previous report: {len(from_prev)}")
print(f"  Newly extracted from R2:     {len(successful_r2)}")
print(f"  Failed / no duration:        {len(failed_r2)}")
print(f"  Total ready to update:       {total_successful}")

# Show all successful extractions
all_successful = [
    *[(r, r["extracted_duration"]) for r in from_prev],
    *successful_r2,
]

if all_successful:
    print(f"\n{'─' * 80}")
    print(f"SUCCESSFUL EXTRACTIONS (first 40)")
    print(f"{'─' * 80}")
    rows = []
    for r, dur in sorted(all_successful, key=lambda x: x[0].get("original_filename", ""))[:40]:
        rows.append([
            r.get("original_filename", "—")[:35],
            fmt_duration(dur),
            f"{dur:.2f}s",
            round(dur),
        ])
    print(tabulate(rows, headers=["Filename", "Duration", "Exact (s)", "Rounded (s)"], tablefmt="simple"))
    if len(all_successful) > 40:
        print(f"\n  ... and {len(all_successful) - 40} more.")

if failed_r2:
    print(f"\n{'─' * 80}")
    print(f"FAILED EXTRACTIONS ({len(failed_r2)} files)")
    print(f"{'─' * 80}")
    rows = []
    for r, _ in sorted(failed_r2, key=lambda x: x[0].get("original_filename", "")):
        size_mb = f"{r['file_size_bytes'] / 1024 / 1024:.1f} MB" if r.get("file_size_bytes") else "—"
        rows.append([
            r.get("original_filename", "—")[:35],
            r.get("media_type", "—"),
            size_mb,
            r["storage_path"][:45],
        ])
    print(tabulate(rows, headers=["Filename", "Type", "Size", "Storage Path"], tablefmt="simple"))

DURATION EXTRACTION RESULTS
  Reused from previous report: 0
  Newly extracted from R2:     145
  Failed / no duration:        0
  Total ready to update:       145

────────────────────────────────────────────────────────────────────────────────
SUCCESSFUL EXTRACTIONS (first 40)
────────────────────────────────────────────────────────────────────────────────
Filename                          Duration    Exact (s)      Rounded (s)
--------------------------------  ----------  -----------  -------------
2024_06_02_20_44_34_IMG_0462.MOV  0:10        10.64s                  11
2024_06_02_22_49_11_IMG_0463.MOV  2:25        145.95s                146
2024_07_01_21_53_50_IMG_0847.MP4  0:17        17.17s                  17
2024_07_04_10_14_14_IMG_0911.MP4  1:21        81.82s                  82
2024_07_09_19_54_25_IMG_0975.MOV  0:14        14.95s                  15
2024_08_02_12_40_49_IMG_1425.MOV  0:14        14.81s                  15
2024_08_02_13_04_19_IMG_1428.MOV  0:55        55.53s   

---
## Step 5: Export Report to CSV

Saves results to `duration_report.csv`. On re-runs, successful extractions are reused from this file.

In [8]:
import csv

# Build lookup of all extracted durations
extracted_from_prev = {r["id"]: r["extracted_duration"] for r in from_prev}
extracted_from_r2 = {r["id"]: d for r, d in r2_results if d}

with open(REPORT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "id", "original_filename", "storage_path",
        "extracted_duration", "rounded_duration", "status"
    ])

    for r in sorted(missing_duration, key=lambda x: x.get("original_filename", "")):
        rid = r["id"]
        filename = r.get("original_filename", "")
        spath = r.get("storage_path", "")

        if rid in extracted_from_prev:
            dur = extracted_from_prev[rid]
            status = "ready"
        elif rid in extracted_from_r2:
            dur = extracted_from_r2[rid]
            status = "ready"
        else:
            dur = ""
            status = "no_data"

        writer.writerow([
            rid, filename, spath,
            f"{dur:.2f}" if dur != "" else "",
            round(dur) if dur != "" else "",
            status,
        ])

print(f"Exported {len(missing_duration)} rows to: {REPORT_PATH}")
print(f"  Ready: {total_successful}")
print(f"  No data: {len(failed_r2)}")

Exported 145 rows to: C:\Users\minds\Videos\DANCE\dance-library\TESTING\duration_report.csv
  Ready: 145
  No data: 0


---
## Step 6: Apply Updates to Supabase

Reads `duration_report.csv` and updates all records marked `ready`. Stores duration as **rounded integer seconds** (matching the existing schema).

In [9]:
import csv as csv_mod
from pathlib import Path

report_path = Path(r"C:\Users\minds\Videos\DANCE\dance-library\TESTING\duration_report.csv")

to_update = []
with open(report_path, "r", encoding="utf-8") as f:
    for row in csv_mod.DictReader(f):
        if row["status"] == "ready" and row["rounded_duration"]:
            to_update.append(row)

print(f"Found {len(to_update)} records marked 'ready' in report.")
print()

updated = 0
errors = []
for row in to_update:
    rid = row["id"]
    duration_val = int(row["rounded_duration"])

    try:
        sb.table("media").update({"duration": duration_val}).eq("id", rid).execute()
        updated += 1
    except Exception as e:
        errors.append((rid, row.get("original_filename"), str(e)))

print(f"Updated: {updated}")
if errors:
    print(f"\nErrors ({len(errors)}):")
    for rid, fn, err in errors:
        print(f"  {fn} ({rid}): {err}")
else:
    print("No errors.")

Found 145 records marked 'ready' in report.

Updated: 145
No errors.


---
## Step 7: Verify

Quick sanity check — re-query Supabase to confirm durations were written.

In [10]:
# Re-fetch all media and check duration coverage
verify = []
offset = 0
while True:
    res = sb.table("media").select("id, duration, storage_path").range(offset, offset + 999).execute()
    verify.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

with_dur = sum(1 for r in verify if r.get("duration") is not None)
without_dur = sum(1 for r in verify if r.get("duration") is None and r.get("storage_path"))
images = sum(1 for r in verify if r.get("duration") is None and not r.get("storage_path"))

total_videos = with_dur + without_dur

print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"  Total records:           {len(verify)}")
print(f"  Videos with duration:    {with_dur}")
print(f"  Videos without duration: {without_dur}")
print(f"  Images (no video file):  {images}")
if total_videos > 0:
    print(f"  Video coverage:          {with_dur / total_videos * 100:.1f}%")

VERIFICATION
  Total records:           359
  Videos with duration:    359
  Videos without duration: 0
  Images (no video file):  0
  Video coverage:          100.0%
